# Zero-shot benchmark of open models (Colab)
Compares open chat models with the GPT-5.4 baseline (64-68% top-1 on the real-style sets) and with our fine-tuned 1.5B run I (about 30%), using the same 186-name menu.

Upload `fallacy_code.zip` and `external_eval.zip` (private, local use only) to the Files sidebar and wait for both to finish. Needs the `HF_TOKEN` secret with *Notebook access* on.

**GPU:** a T4 (16 GB) runs the 4-bit 7B/8B/9B models. A bigger GPU (L4, A100) is faster and is needed for stable Gemma (Gemma wants bf16; a T4 falls back to fp16).
**Gated models:** the `unsloth/...` copies of Gemma and Llama need no access request. The official `google/...` and `meta-llama/...` repos do (accept their licence on the model page first).
**Time (rough guess, not measured):** 15-30 minutes per model on a T4 including the download. Run the smoke test first.

In [ ]:
!nvidia-smi
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm huggingface_hub bitsandbytes sentencepiece

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()  # the secret can only be read from the notebook process
!ls -la /content/*.zip
!rm -rf /content/work && mkdir -p /content/work
!for z in /content/fallacy_code.zip /content/external_eval.zip; do unzip -tq $z || echo "BROKEN (incomplete upload): $z"; unzip -oq $z -d /content/work; done
%cd /content/work
!python fetch_data.py --rev v5
!wc -l data/generated/*.jsonl external/*.jsonl

## Smoke test (5 texts): must print a result line without errors

In [ ]:
!python zeroshot_open.py --model Qwen/Qwen2.5-7B-Instruct --four-bit --limit 5 --sets eval

## Qwen2.5-7B-Instruct

In [ ]:
!python zeroshot_open.py --model Qwen/Qwen2.5-7B-Instruct --four-bit 2>&1 | grep -v 'it/s\]' | tee zs_qwen7b.log

## Qwen2.5-14B-Instruct (about 2x slower)

In [ ]:
!python zeroshot_open.py --model Qwen/Qwen2.5-14B-Instruct --four-bit 2>&1 | grep -v 'it/s\]' | tee zs_qwen14b.log

## Gemma-2-9B-it (best on an L4/A100; on a T4 it may give NaNs or garbage, check `unparsed` in the output)

In [ ]:
!python zeroshot_open.py --model unsloth/gemma-2-9b-it --four-bit 2>&1 | grep -v 'it/s\]' | tee zs_gemma9b.log

## Llama-3.1-8B-Instruct

In [ ]:
!python zeroshot_open.py --model unsloth/Llama-3.1-8B-Instruct --four-bit 2>&1 | grep -v 'it/s\]' | tee zs_llama8b.log

## Download the results

In [ ]:
!zip -q zeroshot_results.zip data/zeroshot_*.json zs_*.log
from google.colab import files
files.download('/content/work/zeroshot_results.zip')